# 간기능이상 EDA
**데이터**: hn_all_preprocessed_v2.csv  
**목적**: 서비스용 간기능이상 스크리닝 모델 — 타겟 파생 전 데이터 현황 파악  
**순서**: 데이터 로드 → 컬럼 탐색 → 간 관련 변수 확인 → 타겟 파생 → 분포/결측/상관 분석

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

DATA_PATH = "/Users/admin/PycharmProjects/AH_03_03/ai_worker/data/hn_all_preprocessed_v2.csv"

df = pd.read_csv(DATA_PATH)
print(f"shape: {df.shape}")
df.head(3)

shape: (19765, 31)


,성별,나이,현재흡연,걷기일수,근력운동일수,고혈압가족력_부,고혈압가족력_모,고혈압가족력_형제,고지혈증가족력_부,고지혈증가족력_모,...,직업_기능노무,직업_주부학생,직업_무직,직업_작업미상,고혈압유병,당뇨유병,이상지질혈증유병,과거음주_현재금주,음주빈도_enc,음주량_enc
0,2.0,56.0,0.0,5.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0,0,1,0,1.0,0.0,0.0,0,1,0.0
1,1.0,30.0,0.0,8.0,6.0,1.0,1.0,0.0,1.0,0.0,...,0,0,0,0,0.0,0.0,0.0,0,3,2.0
2,2.0,25.0,0.0,8.0,6.0,1.0,1.0,1.0,1.0,1.0,...,0,0,1,0,0.0,0.0,0.0,0,4,1.0


## 1. 전체 컬럼 탐색

In [2]:
# 전체 컬럼 목록
print(f"총 컬럼 수: {len(df.columns)}")
print("\n전체 컬럼:")
for i, col in enumerate(df.columns):
    print(f"  [{i:3d}] {col}")

총 컬럼 수: 31

전체 컬럼:
  [  0] 성별
  [  1] 나이
  [  2] 현재흡연
  [  3] 걷기일수
  [  4] 근력운동일수
  [  5] 고혈압가족력_부
  [  6] 고혈압가족력_모
  [  7] 고혈압가족력_형제
  [  8] 고지혈증가족력_부
  [  9] 고지혈증가족력_모
  [ 10] 고지혈증가족력_형제
  [ 11] 당뇨가족력_부
  [ 12] 당뇨가족력_모
  [ 13] 당뇨가족력_형제
  [ 14] 키
  [ 15] 체중
  [ 16] BMI
  [ 17] 직업_관리전문
  [ 18] 직업_사무
  [ 19] 직업_서비스판매
  [ 20] 직업_농림어업
  [ 21] 직업_기능노무
  [ 22] 직업_주부학생
  [ 23] 직업_무직
  [ 24] 직업_작업미상
  [ 25] 고혈압유병
  [ 26] 당뇨유병
  [ 27] 이상지질혈증유병
  [ 28] 과거음주_현재금주
  [ 29] 음주빈도_enc
  [ 30] 음주량_enc


In [3]:
# 간 관련 변수 자동 탐색
liver_keywords = ['ast', 'alt', 'ggt', 'hepa', 'liver', 'bili', 'alp', 'DI1', 'di1']
liver_cols = [c for c in df.columns if any(k.lower() in c.lower() for k in liver_keywords)]
print("간 관련 후보 컬럼:")
for c in liver_cols:
    print(f"  {c}")

간 관련 후보 컬럼:


In [4]:
# HE_ 계열 전체 확인
he_cols = [c for c in df.columns if c.upper().startswith('HE_')]
print(f"HE_ 계열 컬럼 수: {len(he_cols)}")
print(he_cols)

HE_ 계열 컬럼 수: 0
[]


In [5]:
# DI (진단코드) 계열 확인
di_cols = [c for c in df.columns if c.upper().startswith('DI')]
print(f"DI_ 계열 컬럼 수: {len(di_cols)}")
print(di_cols)

DI_ 계열 컬럼 수: 0
[]


## 2. 간 핵심 변수 존재 여부 확인

In [6]:
# 핵심 변수 체크리스트
check_vars = {
    'HE_ast':   'AST (간세포 손상)',
    'HE_alt':   'ALT (간세포 손상 — 간 특이적)',
    'HE_ggt':   'GGT (담즙정체/음주)',
    'HE_hepaB': 'B형간염 표면항원',
    'HE_hepaC': 'C형간염 항체',
    'DI1_pr':   '간질환 의사 진단 이력',
}

print("=" * 55)
print(f"{'변수명':<12} {'설명':<25} {'존재여부'}") 
print("=" * 55)
for var, desc in check_vars.items():
    exists = var in df.columns
    status = '✅ 있음' if exists else '❌ 없음'
    print(f"{var:<12} {desc:<25} {status}")
print("=" * 55)

변수명          설명                        존재여부
HE_ast       AST (간세포 손상)              ❌ 없음
HE_alt       ALT (간세포 손상 — 간 특이적)      ❌ 없음
HE_ggt       GGT (담즙정체/음주)             ❌ 없음
HE_hepaB     B형간염 표면항원                 ❌ 없음
HE_hepaC     C형간염 항체                   ❌ 없음
DI1_pr       간질환 의사 진단 이력              ❌ 없음


## 3. 핵심 변수 기초 통계 & 결측률

In [7]:
# 실제 존재하는 핵심 변수만 필터링
core_vars = [v for v in check_vars.keys() if v in df.columns]
print(f"실제 사용 가능한 핵심 변수: {core_vars}")

# 기초 통계
df[core_vars].describe().T

실제 사용 가능한 핵심 변수: []


ValueError: Cannot describe a DataFrame without columns

In [ ]:
# 결측률
missing = df[core_vars].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    '결측 수': missing,
    '결측률(%)': missing_pct
})
print(missing_df)

# 시각화
fig, ax = plt.subplots(figsize=(8, 4))
missing_pct[missing_pct > 0].sort_values(ascending=True).plot(
    kind='barh', ax=ax, color='#E74C3C'
)
ax.set_title('핵심 변수 결측률 (%)')
ax.set_xlabel('결측률 (%)')
plt.tight_layout()
plt.show()

## 4. AST / ALT / GGT 분포

In [ ]:
enzyme_vars = [v for v in ['HE_ast', 'HE_alt', 'HE_ggt'] if v in df.columns]

if enzyme_vars:
    fig, axes = plt.subplots(1, len(enzyme_vars), figsize=(5 * len(enzyme_vars), 4))
    if len(enzyme_vars) == 1:
        axes = [axes]

    cutoffs = {'HE_ast': 40, 'HE_alt': 40, 'HE_ggt': 60}

    for ax, var in zip(axes, enzyme_vars):
        data = df[var].dropna()
        # 극단값 제거 후 시각화 (99th percentile)
        cap = data.quantile(0.99)
        data_capped = data[data <= cap]
        
        ax.hist(data_capped, bins=60, color='#3498DB', alpha=0.7, edgecolor='white')
        if var in cutoffs:
            ax.axvline(cutoffs[var], color='red', linestyle='--', linewidth=1.5,
                       label=f'컷오프 {cutoffs[var]}')
        ax.set_title(f'{var} 분포 (99th pct 이하)')
        ax.set_xlabel('U/L')
        ax.legend()
    
    plt.tight_layout()
    plt.show()

    # 컷오프 초과 비율
    print("\n컷오프 초과 비율:")
    for var in enzyme_vars:
        if var in cutoffs:
            total = df[var].notna().sum()
            over = (df[var] > cutoffs[var]).sum()
            print(f"  {var} > {cutoffs[var]}: {over:,}명 / {total:,}명 ({over/total*100:.1f}%)")

## 5. 바이러스 간염 / 진단코드 분포

In [ ]:
categorical_vars = [v for v in ['HE_hepaB', 'HE_hepaC', 'DI1_pr'] if v in df.columns]

for var in categorical_vars:
    print(f"\n[{var}] value_counts:")
    vc = df[var].value_counts(dropna=False)
    pct = (vc / len(df) * 100).round(2)
    print(pd.DataFrame({'count': vc, 'pct(%)': pct}))

## 6. 타겟 파생 (복합 OR 조건)

In [ ]:
# 조건별 플래그 생성 (존재하는 변수만)
conds = []

if 'HE_ast' in df.columns:
    conds.append(df['HE_ast'] > 40)
if 'HE_alt' in df.columns:
    conds.append(df['HE_alt'] > 40)
if 'HE_ggt' in df.columns:
    conds.append(df['HE_ggt'] > 60)
if 'HE_hepaB' in df.columns:
    conds.append(df['HE_hepaB'] == 1)
if 'HE_hepaC' in df.columns:
    conds.append(df['HE_hepaC'] == 1)
if 'DI1_pr' in df.columns:
    conds.append(df['DI1_pr'] == 1)

if conds:
    combined = conds[0]
    for c in conds[1:]:
        combined = combined | c
    df['target_liver'] = combined.astype(int)
else:
    print("⚠️ 타겟 파생 가능한 변수 없음 — 컬럼명 재확인 필요")

# 타겟 분포
if 'target_liver' in df.columns:
    vc = df['target_liver'].value_counts()
    pct = (vc / len(df) * 100).round(2)
    print("\n타겟 분포 (target_liver):")
    print(pd.DataFrame({'count': vc, 'pct(%)': pct}))

    # 파이차트
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(vc, labels=['정상(0)', '이상(1)'], autopct='%1.1f%%',
           colors=['#2ECC71', '#E74C3C'], startangle=90)
    ax.set_title('target_liver 클래스 분포')
    plt.tight_layout()
    plt.show()

In [ ]:
# 조건별 기여도 분석
cond_map = {}
if 'HE_ast' in df.columns:   cond_map['AST>40']      = (df['HE_ast'] > 40)
if 'HE_alt' in df.columns:   cond_map['ALT>40']      = (df['HE_alt'] > 40)
if 'HE_ggt' in df.columns:   cond_map['GGT>60']      = (df['HE_ggt'] > 60)
if 'HE_hepaB' in df.columns: cond_map['HBsAg양성']   = (df['HE_hepaB'] == 1)
if 'HE_hepaC' in df.columns: cond_map['HCV양성']     = (df['HE_hepaC'] == 1)
if 'DI1_pr' in df.columns:   cond_map['간질환진단']  = (df['DI1_pr'] == 1)

print("조건별 해당자 수:")
for name, cond in cond_map.items():
    n = cond.sum()
    print(f"  {name:<12}: {n:,}명 ({n/len(df)*100:.1f}%)")

## 7. 피처 후보군 결측률 전체 현황

In [ ]:
# HE_ 계열 전체 결측률
he_cols = [c for c in df.columns if c.upper().startswith('HE_')]

if he_cols:
    miss = df[he_cols].isnull().mean().sort_values(ascending=False) * 100
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(he_cols) * 0.3)))
    miss.plot(kind='barh', ax=ax, color='#E67E22', alpha=0.8)
    ax.axvline(30, color='red', linestyle='--', linewidth=1, label='30% 기준선')
    ax.set_title('HE_ 계열 변수 결측률')
    ax.set_xlabel('결측률 (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # 결측률 30% 초과 변수 경고
    high_miss = miss[miss > 30]
    if len(high_miss) > 0:
        print(f"\n⚠️ 결측률 30% 초과 변수 ({len(high_miss)}개) — 피처 제외 검토 필요:")
        print(high_miss.round(1))
    else:
        print("\n✅ 결측률 30% 초과 변수 없음")

## 8. 타겟별 핵심 변수 분포 비교

In [ ]:
if 'target_liver' in df.columns:
    compare_vars = [v for v in ['HE_ast', 'HE_alt', 'HE_ggt'] if v in df.columns]
    
    if compare_vars:
        fig, axes = plt.subplots(1, len(compare_vars), figsize=(5 * len(compare_vars), 4))
        if len(compare_vars) == 1:
            axes = [axes]

        for ax, var in zip(axes, compare_vars):
            cap = df[var].quantile(0.99)
            for label, color in [(0, '#2ECC71'), (1, '#E74C3C')]:
                data = df[df['target_liver'] == label][var].dropna()
                data = data[data <= cap]
                ax.hist(data, bins=50, alpha=0.5, color=color,
                        label=f"{'정상' if label==0 else '이상'}")
            ax.set_title(f'{var} — 타겟별 분포')
            ax.legend()

        plt.tight_layout()
        plt.show()

## 9. 인구통계 변수 확인

In [ ]:
# 나이/성별/소득 계열 탐색
demo_keywords = ['age', 'sex', 'income', 'edu', 'DJ', 'dj']
demo_cols = [c for c in df.columns if any(k.lower() in c.lower() for k in demo_keywords)]
print("인구통계 후보 컬럼:")
print(demo_cols)

if demo_cols:
    df[demo_cols].describe().T

## 10. EDA 요약

In [ ]:
print("=" * 60)
print("EDA 요약")
print("=" * 60)
print(f"총 샘플 수     : {len(df):,}")
print(f"총 컬럼 수     : {len(df.columns)}")
print(f"HE_ 계열 수    : {len([c for c in df.columns if c.upper().startswith('HE_')])}")

if 'target_liver' in df.columns:
    n_pos = df['target_liver'].sum()
    n_neg = len(df) - n_pos
    ratio = n_pos / len(df) * 100
    print(f"타겟 양성(이상): {n_pos:,}명 ({ratio:.1f}%)")
    print(f"타겟 음성(정상): {n_neg:,}명 ({100-ratio:.1f}%)")
    print(f"클래스 불균형  : 약 1:{n_neg//max(n_pos,1)}")

print("=" * 60)
print("다음 단계 체크리스트:")
print("  [ ] 타겟 파생 기준 확정 (위 조건별 기여도 참고)")
print("  [ ] 결측률 30% 초과 변수 제외 여부 결정")
print("  [ ] 피처 최종 선택 (HE_* + 인구통계 + 음주/흡연)")
print("  [ ] 불균형 전략 결정 (class_weight / SMOTE / threshold)")
print("=" * 60)